# N167 · 死锁、饥饿与哲学家问题

**目标：** 区分安全性、无死锁和有条件的无饥饿。

**先修：** N164, N165, N166, N013。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 资源顺序；循环等待；FIFO准入；公平调度假设；有限持有时间；活性。

**配套讲解来源：** [同名逐章意见](../../comment/167_deadlocks_fairness_philosophers.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

经典"哲学家就餐"问题（LeetCode 1226）：5 位哲学家围一桌，5 把叉子放在相邻两人之间，哲学家 i 的左右叉分别是 `i` 和 `(i+1)%5`。任意线程可以随时调用 `wantsToEat(philosopher, ...)`，方法体内按顺序执行六个回调：拿左叉、拿右叉、吃、放左叉、放右叉。你要保证：

1. **互斥（安全）**：一把叉子同一时刻最多属于一个人，谁也不能和邻居同时开吃。
2. **无死锁**：不会出现"人人心怀一把叉、个个等着另一把"的循环等待，谁也吃不上。
3. **无饥饿（公平）**：只要每个人都有限时长地吃，先来排队的请求不会被后来者无限插队。

三者是递进关系，本章标题"区分安全性、无死锁和有条件的无饥饿"说的就是：**没有死锁不等于没有饥饿**。一个反例直觉：如果只用"给叉子编号、按从小到大拿"的统一顺序，确实无死锁，但某位哲学家可能永远抢不过手快的邻居——这就是"安全且无死锁但会饿"。

举个具体数字例子：哲学家 0 和 2 不相邻（0 用叉 0、1；2 用叉 2、3），他们的请求互不冲突可以先后处理；而哲学家 4 的右叉是 `(4+1)%5=0`，和哲学家 0 的左叉是同一把叉 0，所以他俩必须错开。本章方案是"FIFO 排队 + 一把锁内同时预留两把叉"，并提供 `validate_fork_trace` 检查器重放真实轨迹来验证全部约束。

## 2. 基础知识：先读懂这些词

- **循环等待（circular wait）**：每个线程都占着一把叉、等着邻居手里那把，5 个人围成一圈互相等，谁也动不了——这是死锁的经典形态。成因是"允许拿着一把叉等另一把"。
- **资源有序分配**：给资源编号，规定必须按编号从小到大申请，破坏循环等待的条件。它能保证无死锁，但**不自动保证无饥饿**（练习 1 的主题）。
- **FIFO 准入（先来先服务）**：所有就餐请求按到达顺序排成一条队列，只有队头请求有权拿叉。后来的请求就算两把叉都空闲也必须排队，从制度上杜绝插队饿人。
- **请求票号（ticket）**：每次 `wantsToEat` 调用领到的唯一号码。**队列的单位是票号而不是哲学家编号**——“思路推导”部分 特意强调这一点：同一个哲学家可以并发发起多次调用，每次调用都要有独立的等待、授予和释放记录，用哲学家编号当单位会把它们搅在一起。
- **条件变量（Condition）**：一把锁加等待队列。本章所有状态（等待队列、next_ticket、owners）都在同一把锁内查看和修改，`wait()` 放锁睡觉、`notify_all()` 叫醒众人重查。
- **预留式授予（同时预留两叉）**：在同一互斥区内一次性检查并占住左右两把叉，中间没有放锁的缝隙——因此不可能出现"占住左叉等右叉"，循环等待从根上不存在。
- **队头阻塞（head-of-line blocking）**：严格 FIFO 的代价——队头请求的两把叉没空时，即使队尾某请求的两把叉都空着也得等。这是"可解释的公平性"对吞吐量的让步，是取舍而不是 bug。
- **有限持有时间**：每个人吃一口就放叉（回调有限完成）。这是无饥饿论证的前提之一：前面的请求占用的资源有限时间内必然归还。
- **轨迹检查器（validate_fork_trace）**：把运行期记录的事件流（request/grant/pick/eat/put/release）在单线程里重放一遍，逐条断言互斥、顺序、成对拿放等约束——用"事后审计"代替"相信并发跑完没炸"。

## 3. 思路推导：从小例子开始

**Step 1：把"吃一口饭"建模成一条事件链。** 每个请求产生 8 个事件：`request`（领票排队）→ `grant`（同时拿到两叉的所有权）→ `pick`（左叉）→ `pick`（右叉）→ `eat` → `put`（左叉）→ `put`（右叉）→ `release`（归还所有权）。检查器就按这条链审计。

**Step 2：领票与排队。** `wantsToEat` 一进互斥区就领票：`ticket = self.next_ticket`，自增计数器，把自己 `(ticket, philosopher)` 追加到 `waiting` 队尾，然后 `notify_all`（告知可能正在吃的人"又有人排队了"，虽然他们多半不关心，通知是让等待逻辑统一）。

**Step 3：授予条件是"我是队头且两叉全空"。** 线程在 `while self.waiting[0][0] != ticket or self.owners[left] is not None or self.owners[right] is not None:` 上等待——三个条件缺一不可：不是队头就等（FIFO），左叉或右叉被占也等（互斥）。醒来后把自己弹出队头、把两把叉的 owner 都写成自己的票号，这个"检查+占用"在锁内一气呵成。

**Step 4：吃饭与释放。** 拿到授权后按回调顺序执行拿叉、吃、放叉（每个回调后补记一个事件），`finally` 里无条件归还两把叉的所有权并 `notify_all`——用 try/finally 保证即使回调抛异常也会释放叉子，不留"带着叉子逃跑"的窗口。

**Step 5：手推 “运行示例”部分 的小实例。** 5 位哲学家 `[0,2,4,1,3]` 各发起一次请求（回调全是空操作），假设领票顺序恰好是启动顺序：

```
票0(p0,叉0,1) 等待时两叉全空 → 立即授予 → pick,pick,eat,put,put → release
票1(p2,叉2,3) 两叉全空 → 授予 → 吃完释放
票2(p4,叉4,0) 叉0刚被票0归还 → 空闲 → 授予 → 吃完释放
票3(p1,叉1,2) → 授予 → 释放；票4(p3,叉3,4) → 授予 → 释放
```

“运行示例”部分 的表格就是这 5×8=40 条事件的实录，最后 `validate_fork_trace(obj.events)` 通过。注意票 2 依赖票 0 归还叉 0——这就是"哲学家 4 与哲学家 0 共享叉 0"的体现。

**Step 6：检查器怎么审计。** `validate_fork_trace` 维护自己的重放状态：`waiting` 队列、每票的 `records`、`owners` 数组。每读一个事件就断言它在该发生的位置发生：grant 必须弹出队头且两叉原主为 None；pick 必须先有 grant 且叉主是自己；eat 必须两叉都 pick 过、一把没 put；release 必须两叉都 put 过才归还。结尾要求队列清空、全部叉子无主、每票都 released。

## 4. 核心代码：DiningPhilosophersFIFO

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
class DiningPhilosophersFIFO:

    def __init__(self):
        self.condition = threading.Condition()
        self.waiting = deque()
        self.next_ticket = 0
        self.owners = [None] * 5
        self.events = []

    def _event(self, action, ticket, philosopher, payload=None):
        self.events.append((action, ticket, philosopher, payload))

    def wantsToEat(self, philosopher, pickLeftFork, pickRightFork, eat, putLeftFork, putRightFork):
        if philosopher not in range(5):
            raise ValueError('Philosopher must be 0..4')
        left, right = (philosopher, (philosopher + 1) % 5)
        with self.condition:
            ticket = self.next_ticket
            self.next_ticket += 1
            self.waiting.append((ticket, philosopher))
            self._event('request', ticket, philosopher)
            self.condition.notify_all()
            while self.waiting[0][0] != ticket or self.owners[left] is not None or self.owners[right] is not None:
                self.condition.wait()
            self.waiting.popleft()
            self.owners[left] = self.owners[right] = ticket
            self._event('grant', ticket, philosopher, (left, right))
            self.condition.notify_all()
        try:
            pickLeftFork()
            with self.condition:
                self._event('pick', ticket, philosopher, left)
            pickRightFork()
            with self.condition:
                self._event('pick', ticket, philosopher, right)
            eat()
            with self.condition:
                self._event('eat', ticket, philosopher)
            putLeftFork()
            with self.condition:
                self._event('put', ticket, philosopher, left)
            putRightFork()
            with self.condition:
                self._event('put', ticket, philosopher, right)
        finally:
            with self.condition:
                self.owners[left] = self.owners[right] = None
                self._event('release', ticket, philosopher, (left, right))
                self.condition.notify_all()
        return ticket
```

### 逐段读懂代码

`run_threads` 与前几章相同，略。核心两个部件。

**DiningPhilosophersFIFO：**

（对应程序见下方分段实现与完整代码。）

逐段看：构造函数里 `owners=[None]*5` 记录每把叉当前属于哪个票号，`events` 收集轨迹。入口先校验哲学家编号在 0..4。`left,right = philosopher,(philosopher+1)%5` 算出左右叉——`%5` 让 4 号的右叉绕回 0 号叉。第一段 `with` 是排队与授予：领票、入队、记 request、通知；那个长 `while` 就是授予谓词（队头 + 两叉空闲），不满足就 `wait()`；满足则弹队头、一口气把两把叉的 owner 写成自己（预留式授予），记 grant 再通知。中间 try 块执行六个回调，每个回调后单独拿锁补记事件——注意回调本身在锁外执行，避免拿着全局锁跑用户代码。`finally` 保证无论回调是否抛异常都归还两叉、记 release、通知下一位。返回票号方便测试对账。

**validate_fork_trace（重放检查器）：**

（对应程序见下方分段实现与完整代码。）

它用独立的状态重放事件流：`waiting` 是期望的 FIFO 授予顺序，`records[ticket]` 是每票的生命周期账本，`owners` 重演叉子归属。request 时要求票号唯一并建账本；grant 时断言"弹出的队头恰好是这张票"（FIFO 的直接验证）、两叉原主为 None（互斥）然后占叉；pick 断言先授权、叉主是自己、没重复拿；eat 断言两叉齐、还没放、只吃一次；put 断言吃过且没重复放；release 断言两叉都放过后才清空归属。结尾三连断言：队列清空（每个请求都被授予）、叉子全部无主、每票都完整走完。任何一条不满足都会 AssertionError。

## 5. 分段实现：按顺序运行

**本章接口：** `DiningPhilosophersFIFO.wantsToEat(...)`、`validate_fork_trace(events)`

### THREAD_SOURCE

In [1]:
THREAD_SOURCE = r'''import threading
from collections import deque

def run_threads(functions):
    """Caller must run potentially blocking cases inside the isolated process."""
    errors = []
    error_lock = threading.Lock()
    def guarded(fn):
        try:
            fn()
        except BaseException as error:
            with error_lock:
                errors.append(error)
    threads = [threading.Thread(target=guarded, args=(fn,)) for fn in functions]
    for thread in threads:
        thread.start()
    for thread in threads:
        thread.join()
    if errors:
        raise errors[0]
    assert not any(thread.is_alive() for thread in threads)

class DiningPhilosophersFIFO:
    def __init__(self):
        self.condition = threading.Condition()
        self.waiting = deque()
        self.next_ticket = 0
        self.owners = [None]*5
        self.events = []
    def _event(self,action,ticket,philosopher,payload=None):
        self.events.append((action,ticket,philosopher,payload))
    def wantsToEat(self,philosopher,pickLeftFork,pickRightFork,eat,putLeftFork,putRightFork):
        if philosopher not in range(5):
            raise ValueError('Philosopher must be 0..4')
        left,right = philosopher,(philosopher+1)%5
        with self.condition:
            ticket = self.next_ticket
            self.next_ticket += 1
            self.waiting.append((ticket,philosopher))
            self._event('request',ticket,philosopher)
            self.condition.notify_all()
            while self.waiting[0][0] != ticket or self.owners[left] is not None or self.owners[right] is not None:
                self.condition.wait()
            self.waiting.popleft()
            self.owners[left] = self.owners[right] = ticket
            self._event('grant',ticket,philosopher,(left,right))
            self.condition.notify_all()
        try:
            pickLeftFork()
            with self.condition:self._event('pick',ticket,philosopher,left)
            pickRightFork()
            with self.condition:self._event('pick',ticket,philosopher,right)
            eat()
            with self.condition:self._event('eat',ticket,philosopher)
            putLeftFork()
            with self.condition:self._event('put',ticket,philosopher,left)
            putRightFork()
            with self.condition:self._event('put',ticket,philosopher,right)
        finally:
            with self.condition:
                self.owners[left] = self.owners[right] = None
                self._event('release',ticket,philosopher,(left,right))
                self.condition.notify_all()
        return ticket

def validate_fork_trace(events):
    waiting = deque()
    records = {}
    owners = [None]*5
    for action,ticket,p,payload in events:
        forks = {p,(p+1)%5}
        if action=='request':
            assert ticket not in records
            records[ticket] = {'p':p,'picked':set(),'put':set(),'eaten':0,'released':False,'granted':False}
            waiting.append(ticket)
            continue
        assert ticket in records and records[ticket]['p']==p
        record = records[ticket]
        assert not record['released']
        if action=='grant':
            assert waiting and waiting.popleft()==ticket
            assert not record['granted'] and set(payload)==forks
            assert all(owners[f] is None for f in forks)
            for f in forks:owners[f]=ticket
            record['granted']=True
        elif action=='pick':
            assert record['granted'] and payload in forks and owners[payload]==ticket
            assert payload not in record['picked']
            record['picked'].add(payload)
        elif action=='eat':
            assert record['picked']==forks and not record['put'] and record['eaten']==0
            assert all(owners[f]==ticket for f in forks)
            record['eaten']=1
        elif action=='put':
            assert record['eaten']==1 and payload in forks and payload not in record['put']
            assert owners[payload]==ticket
            record['put'].add(payload)
        elif action=='release':
            assert record['put']==forks and record['eaten']==1 and set(payload)==forks
            assert all(owners[f]==ticket for f in forks)
            for f in forks:owners[f]=None
            record['released']=True
        else:
            raise AssertionError('Unknown trace event')
    assert not waiting and all(x is None for x in owners)
    assert all(r['released'] for r in records.values())
    return True
'''

### 导入本章使用的库

In [2]:
import json
import subprocess
import sys

### 初始化与接口绑定

In [3]:
exec(THREAD_SOURCE, globals())

### run_isolated

In [4]:
def run_isolated(extra):
    """A timeout is a failure; subprocess.run kills and reaps the child."""
    result = subprocess.run([sys.executable, '-c', THREAD_SOURCE + '\n' + extra], text=True, capture_output=True, check=True, timeout=20)
    return result.stdout

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [5]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

rows = json.loads(run_isolated("""import json
obj=DiningPhilosophersFIFO()
callbacks=[lambda:None]*5
run_threads([lambda p=p:obj.wantsToEat(p,*callbacks) for p in [0,2,4,1,3]])
assert validate_fork_trace(obj.events)
print(json.dumps(obj.events))
"""))
show_table(['event','ticket','philosopher','forks'],rows)

event,ticket,philosopher,forks
request,0,0,None
grant,0,0,"[0, 1]"
pick,0,0,0
pick,0,0,1
eat,0,0,None
put,0,0,0
put,0,0,1
release,0,0,"[0, 1]"
request,1,2,None
grant,1,2,"[2, 3]"


## 7. 正确性、适用条件与复杂度

安全性：grant 在同一锁内同时检查并设置两个 owner，任何叉子不会属于两个票号。无死锁：已获准请求不再等待叉子；它有限完成后释放，队头最终可进。无饥饿需要有限回调、锁与线程弱公平调度，以及每个已排队请求之前仅有有限个请求；按票号归纳，前序请求终会退出。有限压测只检查此次运行，不代替这些活性前提。回调异常会向外传播；轨迹不完整不会被检查器视为成功。

**代价：** 叉子状态 O(1)，FIFO 队列 O(q)。本章为核验保存 O(r) 条事件；条件通知可能唤醒多个等待者，不声称每个请求的总调度开销为 O(1)。

### 展开理解

**安全性（不共叉）**：grant 在同一把锁内同时检查并设置两个 owner，中间没有放锁缝隙，所以一把叉绝不可能同时属于两个票号；pick/eat 阶段叉主始终是自己。**无死锁**：被授予的请求已经握有两把叉，之后只执行有限回调、不再等待任何叉子；它吃完必释放（finally 兜底），于是队头终能推进——没有"拿着叉子等叉子"，循环等待无从谈起。**无饥饿（有条件的）**：按票号归纳——0 号票前面没人，只要叉子空出就能走；假设 k 号票之前的都走完了，轮到 k 号当队头，在叉子有限归还的前提下它也被授予。这个论证依赖 “正确性与复杂度”部分 列出的前提：回调有限完成、锁与线程的弱公平调度、每个请求前只有有限个先到者。有限次压测只能检查"这一次跑没出错"，代替不了这些活性前提。**异常路径**：回调抛异常会向外传播（guarded 收集后统一抛出），轨迹不完整的运行不会被检查器判为成功。

**复杂度**：叉子归属是长度 5 的数组，O(1)；FIFO 队列存 q 个排队请求，O(q)；为了核验保存 r 条事件，O(r)。`notify_all` 可能一次唤醒多个等待者，所以本章不声称"每个请求的调度开销 O(1)"——公平与简单是用唤醒风暴换的。

## 8. 单元测试：每个用例在检查什么

“自动测试”部分 在子进程里跑了 3 轮，每轮的构造很讲究：

- `requests=[0,0,0,1,2,3,4,0,2,4]`——哲学家 0 出现 4 次（并发重复请求，验证"票号而非哲学家编号"这条契约），其余各 1 到 2 次。0 号请求（i==0）在 eat 回调里设 `first_started` 并卡在 `release_first.wait()`——人为制造"队头抱着叉子慢慢吃"。
- `coordinate` 协调线程等队头开吃后放行其他请求（`go.set()`），再等 `next_ticket` 涨到 10（全部领完票）才让队头放叉——保证测试窗口内确实形成了一条有 10 张票的完整队列。
- 断言一：`validate_fork_trace(obj.events)` 通过——互斥、FIFO、成对拿放全部审计过关。
- 断言二：`seen==Counter({i:1 for i in range(10)})`——10 个请求每个都吃到了一次（无饥饿的直接证据）。
- 断言三：`len(set(tickets))==10`——票号全不重复。
- **负向测试**：把事件流复制一份，在第 1 位插入一条伪造的 `('eat',0,0,None)`（让 0 号票没拿叉就吃），断言检查器**必须拒绝**它。这一条防的是"检查器是个只会点头的摆设"。
- 收尾 `threading.active_count()==1`：全部线程回收，没人卡在 `condition.wait()`。

In [6]:
print(run_isolated("from collections import Counter\nfor _ in range(3):\n    obj=DiningPhilosophersFIFO()\n    first_started=threading.Event();release_first=threading.Event();go=threading.Event()\n    seen=Counter();seen_lock=threading.Lock();tickets=[]\n    requests=[0,0,0,1,2,3,4,0,2,4]\n    def request(i,p):\n        if i:go.wait()\n        def eating():\n            if i==0:\n                first_started.set();release_first.wait()\n            with seen_lock:seen[i]+=1\n        ticket=obj.wantsToEat(p,lambda:None,lambda:None,eating,lambda:None,lambda:None)\n        with seen_lock:tickets.append(ticket)\n    def coordinate():\n        first_started.wait();go.set()\n        with obj.condition:\n            while obj.next_ticket != len(requests):obj.condition.wait()\n        release_first.set()\n    run_threads([lambda i=i,p=p:request(i,p) for i,p in enumerate(requests)]+[coordinate])\n    assert validate_fork_trace(obj.events)\n    assert seen==Counter({i:1 for i in range(len(requests))})\n    assert len(set(tickets))==len(requests)\n    bad=list(obj.events)\n    bad.insert(1,('eat',0,0,None))\n    try:validate_fork_trace(bad)\n    except AssertionError:pass\n    else:raise AssertionError('Eating without forks was accepted')\nassert threading.active_count()==1\nprint('FIFO tickets, concurrent repeated philosophers, ownership and cleanup passed.')\n"))

print('N167: 所有本章断言通过')

FIFO tickets, concurrent repeated philosophers, ownership and cleanup passed.

N167: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 比较统一加锁顺序和FIFO准入：前者无死锁不自动证明无饥饿。

**练习 2：** 为活性明确调度假设。

<details>
<summary>展开思路提示（不是完整答案）</summary>

**练习 1（统一加锁顺序 vs FIFO 准入）**：提示：写一个对照版——每人拿叉必须"先拿编号小的那把"（哲学家 4 因此要先拿叉 0 再拿叉 4）。分析两层：无死锁上，两者都成立（破坏循环等待 vs 不允许持叉等待）；无饥饿上，FIFO 有制度保证，而"从小到大"版本里，若 0 号和 1 号哲学家高频抢叉 1，0 号可能反复失败——构造一个"1 号总在 0 号之前抢到叉 1"的调度叙事即可说明饥饿是可能的，不必真的复现。可以拿"叉 0 空闲但被队头阻塞"的例子同时讲 FIFO 的队头阻塞代价，说明两种方案各让渡了什么。

**练习 2（为活性明确调度假设）**：提示：把"无饥饿"的论证前提一条条写出来：每个 eat/put 回调有限完成（否则 finally 永不到达）；Python 的锁唤醒是弱公平的（睡着的线程最终会被唤醒，但不保证先到先得）；排队请求之前的先到者是有限个。然后对每条前提各构造一个反例世界：如果有人的 eat 回调死循环，后面全部饿死；如果调度器永远不唤醒某个线程，它也饿死。结论是"无饥饿是条件成立下的定理，不是无条件的保证"，能把这些边界说清楚就算达标。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

本章包含真实的 SQL／Shell／JavaScript／线程程序，Python 部分负责创建示例、调用运行时和断言。嵌入的程序源码也在下面完整显示；缺少相应运行时应安装依赖，不能用模拟结果替代。

In [7]:
# 完整实现：本单元格不依赖其他单元格。
THREAD_SOURCE = r'''import threading
from collections import deque

def run_threads(functions):
    """Caller must run potentially blocking cases inside the isolated process."""
    errors = []
    error_lock = threading.Lock()
    def guarded(fn):
        try:
            fn()
        except BaseException as error:
            with error_lock:
                errors.append(error)
    threads = [threading.Thread(target=guarded, args=(fn,)) for fn in functions]
    for thread in threads:
        thread.start()
    for thread in threads:
        thread.join()
    if errors:
        raise errors[0]
    assert not any(thread.is_alive() for thread in threads)

class DiningPhilosophersFIFO:
    def __init__(self):
        self.condition = threading.Condition()
        self.waiting = deque()
        self.next_ticket = 0
        self.owners = [None]*5
        self.events = []
    def _event(self,action,ticket,philosopher,payload=None):
        self.events.append((action,ticket,philosopher,payload))
    def wantsToEat(self,philosopher,pickLeftFork,pickRightFork,eat,putLeftFork,putRightFork):
        if philosopher not in range(5):
            raise ValueError('Philosopher must be 0..4')
        left,right = philosopher,(philosopher+1)%5
        with self.condition:
            ticket = self.next_ticket
            self.next_ticket += 1
            self.waiting.append((ticket,philosopher))
            self._event('request',ticket,philosopher)
            self.condition.notify_all()
            while self.waiting[0][0] != ticket or self.owners[left] is not None or self.owners[right] is not None:
                self.condition.wait()
            self.waiting.popleft()
            self.owners[left] = self.owners[right] = ticket
            self._event('grant',ticket,philosopher,(left,right))
            self.condition.notify_all()
        try:
            pickLeftFork()
            with self.condition:self._event('pick',ticket,philosopher,left)
            pickRightFork()
            with self.condition:self._event('pick',ticket,philosopher,right)
            eat()
            with self.condition:self._event('eat',ticket,philosopher)
            putLeftFork()
            with self.condition:self._event('put',ticket,philosopher,left)
            putRightFork()
            with self.condition:self._event('put',ticket,philosopher,right)
        finally:
            with self.condition:
                self.owners[left] = self.owners[right] = None
                self._event('release',ticket,philosopher,(left,right))
                self.condition.notify_all()
        return ticket

def validate_fork_trace(events):
    waiting = deque()
    records = {}
    owners = [None]*5
    for action,ticket,p,payload in events:
        forks = {p,(p+1)%5}
        if action=='request':
            assert ticket not in records
            records[ticket] = {'p':p,'picked':set(),'put':set(),'eaten':0,'released':False,'granted':False}
            waiting.append(ticket)
            continue
        assert ticket in records and records[ticket]['p']==p
        record = records[ticket]
        assert not record['released']
        if action=='grant':
            assert waiting and waiting.popleft()==ticket
            assert not record['granted'] and set(payload)==forks
            assert all(owners[f] is None for f in forks)
            for f in forks:owners[f]=ticket
            record['granted']=True
        elif action=='pick':
            assert record['granted'] and payload in forks and owners[payload]==ticket
            assert payload not in record['picked']
            record['picked'].add(payload)
        elif action=='eat':
            assert record['picked']==forks and not record['put'] and record['eaten']==0
            assert all(owners[f]==ticket for f in forks)
            record['eaten']=1
        elif action=='put':
            assert record['eaten']==1 and payload in forks and payload not in record['put']
            assert owners[payload]==ticket
            record['put'].add(payload)
        elif action=='release':
            assert record['put']==forks and record['eaten']==1 and set(payload)==forks
            assert all(owners[f]==ticket for f in forks)
            for f in forks:owners[f]=None
            record['released']=True
        else:
            raise AssertionError('Unknown trace event')
    assert not waiting and all(x is None for x in owners)
    assert all(r['released'] for r in records.values())
    return True
'''

import json

import subprocess

import sys

exec(THREAD_SOURCE, globals())

def run_isolated(extra):
    """A timeout is a failure; subprocess.run kills and reaps the child."""
    result = subprocess.run([sys.executable, '-c', THREAD_SOURCE + '\n' + extra], text=True, capture_output=True, check=True, timeout=20)
    return result.stdout

# 示例与回归测试
print(run_isolated("from collections import Counter\nfor _ in range(3):\n    obj=DiningPhilosophersFIFO()\n    first_started=threading.Event();release_first=threading.Event();go=threading.Event()\n    seen=Counter();seen_lock=threading.Lock();tickets=[]\n    requests=[0,0,0,1,2,3,4,0,2,4]\n    def request(i,p):\n        if i:go.wait()\n        def eating():\n            if i==0:\n                first_started.set();release_first.wait()\n            with seen_lock:seen[i]+=1\n        ticket=obj.wantsToEat(p,lambda:None,lambda:None,eating,lambda:None,lambda:None)\n        with seen_lock:tickets.append(ticket)\n    def coordinate():\n        first_started.wait();go.set()\n        with obj.condition:\n            while obj.next_ticket != len(requests):obj.condition.wait()\n        release_first.set()\n    run_threads([lambda i=i,p=p:request(i,p) for i,p in enumerate(requests)]+[coordinate])\n    assert validate_fork_trace(obj.events)\n    assert seen==Counter({i:1 for i in range(len(requests))})\n    assert len(set(tickets))==len(requests)\n    bad=list(obj.events)\n    bad.insert(1,('eat',0,0,None))\n    try:validate_fork_trace(bad)\n    except AssertionError:pass\n    else:raise AssertionError('Eating without forks was accepted')\nassert threading.active_count()==1\nprint('FIFO tickets, concurrent repeated philosophers, ownership and cleanup passed.')\n"))

print('N167: 所有本章断言通过')

FIFO tickets, concurrent repeated philosophers, ownership and cleanup passed.

N167: 所有本章断言通过


## 11. 代表题与迁移

- **1226. The Dining Philosophers（canonical）**：本章的原题——5 哲学家 5 叉子，练的是无死锁结构（本章用"一把锁内同时预留两叉"）与公平准入（FIFO 票号）的完整设计。
- **1195. Fizz Buzz Multithreaded（extension）**：也是多线程按规则轮流打印（3/5/15 的倍数），练的是和本章同源的"轮到谁"状态管理，可视为 N165 交替打印到本章准入控制之间的过渡题。

**题面入口：** [1226](https://leetcode.com/problems/the-dining-philosophers/)

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。